# AI-Driven Infrastructure Anomaly Detector — Exploration Notebook

This notebook demonstrates the core ML pipeline: data generation, model training, anomaly detection, and evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from src.ml_processing.isolation_forest import IsolationForestDetector
from src.ml_processing.holt_winters import HoltWintersDetector
from src.ml_processing.ensemble import EnsembleDetector
from src.ml_processing.drift_detector import DriftDetector
from src.ingestion.feature_engineering import FeatureEngineer
from src.ingestion.models import MetricPoint

## 1. Generate Synthetic Server Metrics

In [ ]:
np.random.seed(42)
n_samples = 2000
t = np.linspace(0, n_samples / 288, n_samples)

cpu = 45 + 20 * np.sin(2 * np.pi * t) + np.random.normal(0, 5, n_samples)
memory = 60 + 10 * np.sin(2 * np.pi * t + 1) + np.random.normal(0, 3, n_samples)
disk_io = 20 + 15 * np.abs(np.sin(2 * np.pi * t * 2)) + np.random.normal(0, 4, n_samples)

cpu = np.clip(cpu, 0, 100)
memory = np.clip(memory, 0, 100)
disk_io = np.clip(disk_io, 0, 100)

data = np.column_stack([cpu, memory, disk_io])

anomaly_indices = np.random.choice(n_samples, 50, replace=False)
data[anomaly_indices, 0] = np.random.uniform(85, 100, 50)

print(f'Data shape: {data.shape}')
print(f'Anomalous samples: {len(anomaly_indices)} / {n_samples}')

## 2. Train Isolation Forest

In [ ]:
if_detector = IsolationForestDetector.__new__(IsolationForestDetector)
if_detector.model_name = 'isolation_forest'
if_detector.contamination = 0.05
if_detector.n_estimators = 100
if_detector.max_samples = 'auto'
if_detector.random_state = 42
if_detector._model = None
if_detector._trained = False
if_detector._feature_count = 0

metrics = if_detector.train(data)
print('Training metrics:', metrics)

## 3. Evaluate Detection Performance

In [ ]:
from sklearn.metrics import classification_report

results = if_detector.batch_detect(data)
predictions = np.array([1 if r.is_anomaly else 0 for r in results])

ground_truth = np.zeros(n_samples)
ground_truth[anomaly_indices] = 1

print(classification_report(ground_truth, predictions, target_names=['Normal', 'Anomaly']))

## 4. Feature Engineering

In [ ]:
fe = FeatureEngineer.__new__(FeatureEngineer)
fe.window_sizes = [5, 15, 60]
fe.feature_names = ['mean', 'std', 'min', 'max', 'skew', 'kurtosis']
fe._windows = {}
fe._max_window = 10000

ts = datetime.now(timezone.utc)
for i in range(n_samples):
    fe.push(MetricPoint(name='cpu', value=float(data[i, 0]), timestamp=ts, server_id='server-01'))

features = fe.compute_features('server-01', 'cpu', window_size=15)
print('Feature vector (15min window):')
for k, v in sorted(features.items()):
    print(f'  {k}: {v:.4f}')

## 5. Drift Detection

In [ ]:
drift_detector = DriftDetector.__new__(DriftDetector)
drift_detector.threshold = 0.15
drift_detector.retrain_on_drift = True
drift_detector._reference_histograms = {}
drift_detector._reference_bin_edges = {}

drift_detector.set_reference('cpu', data[:1000, 0])

result = drift_detector.check_drift('cpu', data[1000:1500, 0])
print(f'Same distribution PSI: {result["psi"]:.4f} ({result["level"]})')

shifted_data = np.random.normal(70, 15, 500)
result2 = drift_detector.check_drift('cpu', shifted_data)
print(f'Shifted distribution PSI: {result2["psi"]:.4f} ({result2["level"]})')

## 6. Holt-Winters Forecasting

In [ ]:
hw_detector = HoltWintersDetector.__new__(HoltWintersDetector)
hw_detector.model_name = 'holt_winters'
hw_detector.seasonal_periods = 12
hw_detector.trend = 'add'
hw_detector.seasonal = 'add'
hw_detector._model = None
hw_detector._fitted = None
hw_detector._residual_std = 0.0
hw_detector._trained = False
hw_detector._data = None

hw_metrics = hw_detector.train(data[:, 0])
print('HW Training:', hw_metrics)

forecast = hw_detector.forecast(steps=24)
if forecast is not None:
    print(f"Forecast: {forecast[:5]}...")